# assignment 2 code stuff
## part 2: networks
### make sure that we use same N = [1000, 5000, 10000] and same SEED = 42 for both docs

save all data as csv files and upload to git

In [ ]:
#install packages
from scipy.integrate import odeint
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import fftpack
from scipy.optimize import minimize
import math
import ipywidgets as widgets
from ipywidgets import interact
import ndlib
import networkx as nx
import ndlib.models.epidemics as ep
import os
import seaborn as sns

#set seed
SEED = 42
np.random.seed(SEED)

In [ ]:
#make networks of different types
#several using the same params
#rerun using N = 1000, 10000, 100.000?
#how do P and M relate? how to make comparable er and sf networks?

# Parameters
NETWORK_SIZES = [1000, 5000, 10000] #iterate across different ns
NUM_NETWORKS = 5 #5 per param constellation for the stochasticity
M = 10 #how many the new nodes in sf connects to
#SEED set at top of doc to 42

output_dir = "networks"
os.makedirs(output_dir, exist_ok=True)

for N in NETWORK_SIZES:
    # Match the expected average degree of ER to that of BA
    P = (2 * M) / (N - 1)
    print(f"\nGenerating networks with N={N}, P={P:.6f}, M={M}")
    # Generate Erdős–Rényi networks
    for i in range(NUM_NETWORKS):
        g = nx.erdos_renyi_graph(
            n=N,
            p=P,
            seed=SEED + i
        )
        filename = os.path.join(output_dir,f"erdos_renyi_N{N}_{i+1}.graphml")
        nx.write_graphml(g, filename)
        print(
            f"Saved {filename}: "
            f"N={g.number_of_nodes()}, "
            f"L={g.number_of_edges()}, "
            f"mean degree={2*g.number_of_edges()/N:.2f}"
        )
    # Generate scale-free Barabási–Albert networks
    for i in range(NUM_NETWORKS):
        g = nx.barabasi_albert_graph(
            n=N,
            m=M,
            seed=SEED + 100 + i
        )

        filename = os.path.join(output_dir,f"scale_free_N{N}_{i+1}.graphml")
        nx.write_graphml(g, filename)
        print(
            f"Saved {filename}: "
            f"N={g.number_of_nodes()}, "
            f"L={g.number_of_edges()}, "
            f"mean degree={2*g.number_of_edges()/N:.2f}"
        )

print("\nAll networks have been generated and saved.")



In [ ]:
#get degree clustering and centrality stats per network and summary
# folders
input_dir = "networks"
output_dir = "network_metrics"
os.makedirs(output_dir, exist_ok=True)

# Experiment parameters
NETWORK_SIZES = [1000, 5000, 10000]
NUM_NETWORKS = 5
NETWORK_TYPES = ["erdos_renyi", "scale_free"]

# Store summary statistics for all networks
summary = []

for N in NETWORK_SIZES:
    for network_type in NETWORK_TYPES:
        for i in range(1, NUM_NETWORKS + 1):

            # Match the naming convention used when saving networks
            filename = f"{network_type}_N{N}_{i}.graphml"
            filepath = os.path.join(input_dir, filename)

            if not os.path.exists(filepath):
                print(f"File not found, skipping: {filepath}")
                continue

            # Load network
            G = nx.read_graphml(filepath)

            # Network-level properties
            n_nodes = G.number_of_nodes()
            n_edges = G.number_of_edges()
            density = nx.density(G)

            # Node-level metrics
            degree = dict(G.degree())
            degree_centrality = nx.degree_centrality(G)
            clustering = nx.clustering(G)

            # Create table with one row per node
            #centrality and clustering are already normalized in nx
            df = pd.DataFrame({
                "node": list(G.nodes()),
                "degree": [degree[n] for n in G.nodes()],
                "degree_centrality": [
                    degree_centrality[n] for n in G.nodes()
                ],
                "clustering_coefficient": [
                    clustering[n] for n in G.nodes()
                ]
            })

            # Save node-level metrics
            output_filename = filename.replace(".graphml", "_metrics.csv")
            output_path = os.path.join(output_dir, output_filename)
            df.to_csv(output_path, index=False)

            # Network-level summary
            summary.append({
                "network_type": network_type,
                "N": n_nodes,
                "replicate": i,
                "L": n_edges,
                "density": density,
                "mean_degree": df["degree"].mean(),
                "mean_degree_centrality":
                    df["degree_centrality"].mean(),
                "mean_clustering":
                    df["clustering_coefficient"].mean()
            })

            print(
                f"Processed {filename}: "
                f"N={n_nodes}, L={n_edges}, "
                f"mean degree centrality="
                f"{df['degree_centrality'].mean():.5f}, "
                f"mean clustering="
                f"{df['clustering_coefficient'].mean():.5f}"
            )

# Save summary of all networks
summary_df = pd.DataFrame(summary)
summary_path = os.path.join(
    output_dir, "network_summary.csv"
)
summary_df.to_csv(summary_path, index=False)

print("\nAll network metrics have been calculated.")
print(f"Summary saved to: {summary_path}")


In [ ]:
#plot stats comparing er and sf and effect of n

# Load network-level summary
input_file = os.path.join("network_metrics", "network_summary.csv")
output_dir = "network_plots"
os.makedirs(output_dir, exist_ok=True)

summary = pd.read_csv(input_file)

# Friendly labels
summary["network_type"] = summary["network_type"].map({
    "erdos_renyi": "ER",
    "scale_free": "SF"
})

# Average over repetitions for each network type and size
metrics = ["mean_degree_centrality","mean_clustering"]

averaged = (
    summary.groupby(["N", "network_type"])[metrics]
    .agg(["mean", "std"])
    .reset_index()
)

print(averaged)

sns.set_theme(style="whitegrid", context="talk")

# Plot mean degree centrality and clustering
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for network_type, group in averaged.groupby("network_type"):
    # Extract columns from the multi-index table
    x = group["N"].to_numpy()

    centrality_mean = group["mean_degree_centrality"]["mean"].to_numpy()
    centrality_std = group["mean_degree_centrality"]["std"].fillna(0).to_numpy()

    clustering_mean = group["mean_clustering"]["mean"].to_numpy()
    clustering_std = group["mean_clustering"]["std"].fillna(0).to_numpy()

    axes[0].errorbar(
        x, centrality_mean, yerr=centrality_std,
        marker="o", capsize=4, linewidth=2,
        label=network_type
    )

    axes[1].errorbar(
        x, clustering_mean, yerr=clustering_std,
        marker="o", capsize=4, linewidth=2,
        label=network_type
    )

# Degree centrality plot
axes[0].set_title("Mean degree centrality")
axes[0].set_xlabel("Number of nodes (N)")
axes[0].set_ylabel("Mean degree centrality")
axes[0].set_xticks([1000, 5000, 10000])
axes[0].ticklabel_format(axis="x", style="plain")
axes[0].legend(title="Network type")

# Clustering plot
axes[1].set_title("Mean clustering coefficient")
axes[1].set_xlabel("Number of nodes (N)")
axes[1].set_ylabel("Mean local clustering coefficient")
axes[1].set_xticks([1000, 5000, 10000])
axes[1].ticklabel_format(axis="x", style="plain")
axes[1].legend(title="Network type")

plt.tight_layout()

plt.savefig(
    os.path.join(output_dir, "ER_vs_SF_by_network_size.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
#apply simple sir to network
#we use already finished sf models i assume
#fixed number of initial infected = 5
### to do: rerun gillespie etc above so the Ns match all through the doc

In [ ]:
#get dataset from sociopatterns
#run tests below only on this network

In [ ]:
# run baseline: randomly assign vaccination
#start with 1 vacc per timestep, then 5 then 10
#vaccination makes k per timestep go s->r
#careful! we only know status if we test so if we distribute randomly we get a certain numberr of i and r also, so no effect


#second baseline: randomly assign vacc (no test/test accuracy 0) according to sf strat
#so per timestep pick random node, vacc random neighbor, iterate until budget per timestep is gone etc

In [ ]:
#how do we implement testing in network?
#accuracy 1: randomly pick k, test, then take those that have status s -> vacc

#stochastic: randomly pick k, test, with probability p= 0.75 or 0.5 pick those that have status s -> vacc    